In [1]:
import pandas as pd

df = pd.read_csv("spotify_songs_2015_2025.csv")

In [2]:
df.head()

,year,artist,title,streams
0,2015,Lord Huron,The Night We Met,3858784170
1,2015,Justin Bieber,Love Yourself,3312873799
2,2015,Twenty One Pilots,Stressed Out,3118770904
3,2015,The Weeknd,The Hills,3112731294
4,2015,Justin Bieber,Sorry,3023709953


In [3]:
print(df.shape)

(10779, 4)


In [5]:
import requests #this is a way to visit a website 
from bs4 import BeautifulSoup #this lets it cut out only the info we want
import pandas as pd 

In [6]:
!pip install beautifulsoup4

In [8]:
url = "https://www.grammy.com/awards/67th-annual-grammy-awards-2024/"
headers = {"User-Agent": "Mozilla/5.0"
          }
response = requests.get(url, headers=headers, timeout = 30)
print("Status code:", response.status_code)
print("Page length:", len(response.text))

Status code: 200
Page length: 502038


In [10]:
soup = BeautifulSoup(response.text, "html.parser")
print(soup.title)

<title>67th Annual Grammy Awards 2025 | Grammy</title>


In [11]:
page_text = soup.get_text()
print("Winner" in page_text)
print("Nominee" in page_text)

True
True


In [12]:
print("Billie" in page_text)

True


In [13]:
#we have to see how the grammy page organizes each category, winner, and nominee
for heading in soup.find_all(["h1", "h2", "h3", "h4"]):
    text = heading.get_text(" ", strip=True)
    if text:
        print(text)

67th annual grammy awards
      •
      2025
      Telecast
Winners
The Grammys are more than just Awards
Advancement
Music Advocacy
Ready to Reach Out? We’re Ready to Listen.
Sign Up For The Gramophone


In [15]:
#check where Billie appears 
billie_match = soup.find_all(string=lambda text: text and "Billie" in text)
print("Number of matches:" , len(billie_match))

for match in billie_match[:5]:
    print("\n---MATCH---")
    print(match.parent.prettify()[:2000])

Number of matches: 6

---MATCH---
<a class="text-body text-decoration-none" href="https://www.grammy.com/artists/billie-eilish/251741/">
 Billie Eilish
</a>


---MATCH---
<span data-no-translation="">
 Billie Eilish &amp; FINNEAS, producers; Thom Beemer, Jon Castelli, Billie Eilish, FINNEAS, Aron Forbes, Brad Lauchert &amp; Chaz Sexton, engineers/mixers; Dale Becker, mastering engineer
</span>


---MATCH---
<a class="text-body text-decoration-none" href="https://www.grammy.com/artists/billie-eilish/251741/">
 Billie Eilish
</a>


---MATCH---
<span data-no-translation="">
 Billie Eilish &amp; FINNEAS, producers; Thom Beemer, Jon Castelli, Billie Eilish, FINNEAS, Aron Forbes, Brad Lauchert &amp; Chaz Sexton, engineers/mixers; Billie Eilish O'Connell &amp; FINNEAS, songwriters; Dale Becker, mastering engineer
</span>


---MATCH---
<a class="text-body text-decoration-none" href="https://www.grammy.com/artists/billie-eilish/251741/">
 Billie Eilish
</a>



In [38]:
import re 
import time 
from io import StringIO

def normalize_text(value):
    if pd.isna(value):
        return ""
    value = str(value).lower()
    value = value.replace("&", "and")
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value)

    return value

In [39]:
#collect grammy year links 

grammy_year_links = {}
for link in soup.find_all("a", href = True):
    text = link.get_text(" ", strip = True)
    href = link["href"]

    if text.isdigit() and "/awards/" in href:
        ceremony_year = int(text)

        if 2015 <= ceremony_year <= 2026:
            if href.startswith("/"):
                href = "https://www.grammy.com/" + href

            grammy_year_links[ceremony_year] = href

print(grammy_year_links)

{2026: 'https://www.grammy.com/awards/68th-annual-grammy-awards-2025/', 2025: 'https://www.grammy.com/awards/67th-annual-grammy-awards-2024/', 2024: 'https://www.grammy.com/awards/66th-annual-grammy-awards-2023/', 2023: 'https://www.grammy.com/awards/65th-annual-grammy-awards-2022/', 2022: 'https://www.grammy.com/awards/64th-annual-grammy-awards-2021/', 2021: 'https://www.grammy.com/awards/63rd-annual-grammy-awards-2020/', 2020: 'https://www.grammy.com/awards/62nd-annual-grammy-awards-2019/', 2019: 'https://www.grammy.com/awards/61st-annual-grammy-awards-2018/', 2018: 'https://www.grammy.com/awards/60th-annual-grammy-awards-2017/', 2017: 'https://www.grammy.com/awards/59th-annual-grammy-awards/', 2016: 'https://www.grammy.com/awards/58th-annual-grammy-awards/', 2015: 'https://www.grammy.com/awards/57th-annual-grammy-awards/'}


In [40]:
import time 
grammy_page_text = {}

for ceremony_year, ceremony_url in sorted(grammy_year_links.items()):
    print("Downloading:", ceremony_year)

    response = requests.get(ceremony_url, headers = headers, timeout = 30)
    if response.status_code == 200:
        ceremony_soup = BeautifulSoup(response.text, "html.parser")
        grammy_page_text[ceremony_year] = (ceremony_soup.get_text(" ", strip = True).lower())

    else:
        print("Failed:", ceremony_year)

    time.sleep(1)

print("Pages downloaded:", len(grammy_page_text))

Downloading: 2015
Downloading: 2016
Downloading: 2017
Downloading: 2018
Downloading: 2019
Downloading: 2020
Downloading: 2021
Downloading: 2022
Downloading: 2023
Downloading: 2024
Downloading: 2025
Downloading: 2026
Pages downloaded: 12


In [41]:
#after debugging, we are going to extract winners from all 12 grammy pages 
from io import StringIO
all_grammy_winners = []

for ceremony_year, ceremony_url in sorted(grammy_year_links.items()):
    print("Reading winners from:", ceremony_year)

    response = requests.get(ceremony_url, headers = headers, timeout = 30)
    if response.status_code != 200:
        print("Failed:", ceremony_year)
        continue

    try:
        tables = pd.read_html(StringIO(response.text))
    except ValueError:
        print("No tables found:", ceremony_year)
        continue

    for table in tables:
        #column names 
        table.columns = [
            str(column).strip().lower()
            for column in table.columns
        ]

        if "category" in table.columns and "work" in table.columns:
            table["ceremony_year"] = ceremony_year

            all_grammy_winners.append(
                table[["ceremony_year", "category", "winner", "work"]]
            )

print("Number of winner tables found:", len(all_grammy_winners))

Reading winners from: 2015
Reading winners from: 2016
Reading winners from: 2017
Reading winners from: 2018
Reading winners from: 2019
Reading winners from: 2020
Reading winners from: 2021
Reading winners from: 2022
Reading winners from: 2023
Reading winners from: 2024
Reading winners from: 2025
Reading winners from: 2026
Number of winner tables found: 12


In [42]:
grammy_winners_df = pd.concat(
    all_grammy_winners, 
    ignore_index = True
)

grammy_winners_df.head(20)

,ceremony_year,category,winner,work
0,2015,"Best Engineered Album, Non-Classical","Drew Brown, Tom Elmhirst, David Greenbaum, Col...",Morning Phase
1,2015,"Best Engineered Album, Classical",Michael J. Bishop,Vaughan Williams: Dona Nobis Pacem; Symphony N...
2,2015,"Producer Of The Year, Classical",Judith Sherman,"Producer Of The Year, Classical"
3,2015,Best Immersive Audio Album,"Elliot Scheiner, Bob Ludwig, Beyoncé Knowles",Beyoncé
4,2015,Best Instrumental Composition,John Williams,The Book Thief
5,2015,"Best Arrangement, Instrumental Or A Cappella","Ben Bram, Pentatonix, Mitch Grassi, Scott Hoyi...",Daft Punk
6,2015,"Best Arrangement, Instruments And Vocals",Billy Childs,New York Tendaberry
7,2015,Record Of The Year,Sam Smith,Stay With Me (Darkchild Version)
8,2015,Album Of The Year,Beck,Morning Phase
9,2015,Song Of The Year,"James Napier, William Phillips, Sam Smith",Stay With Me (Darkchild Version)


In [43]:
grammy_winning_songs = grammy_winners_df[
    ["ceremony_year", "winner", "work"]].drop_duplicates()
grammy_winning_songs.head(20)
print(grammy_winning_songs.shape)

(1022, 3)


In [44]:
df["title_clean"] = df["title"].apply(normalize_text)
df["artist_clean"] = df["artist"].apply(normalize_text)

grammy_winning_songs["title_clean"] = (
    grammy_winning_songs["work"].apply(normalize_text)
)

grammy_winning_songs["winner_clean"] = (
    grammy_winning_songs["winner"].apply(normalize_text)
)

In [45]:
winning_title_set = set(
    grammy_winning_songs["title_clean"]
)

len(winning_title_set)

935

In [46]:
#binary target
df["grammy_winner"] = (
    df["title_clean"]
    .isin(winning_title_set)
    .astype(int)
)
df.head()

,year,artist,title,streams,title_clean,artist_clean,grammy_winner
0,2015,Lord Huron,The Night We Met,3858784170,the night we met,lord huron,0
1,2015,Justin Bieber,Love Yourself,3312873799,love yourself,justin bieber,0
2,2015,Twenty One Pilots,Stressed Out,3118770904,stressed out,twenty one pilots,1
3,2015,The Weeknd,The Hills,3112731294,the hills,the weeknd,0
4,2015,Justin Bieber,Sorry,3023709953,sorry,justin bieber,0


In [47]:
print(df["grammy_winner"].value_counts())

grammy_winner
0    10551
1      228
Name: count, dtype: int64


In [48]:
df[df["grammy_winner"] == 1][
    ["year", "artist", "title"]].head(50)

,year,artist,title
2,2015,Twenty One Pilots,Stressed Out
25,2015,Adele,Hello
28,2015,Drake,Hotline Bling
49,2015,Jack Ü,Where Are Ü Now
68,2015,Kendrick Lamar,Alright
79,2015,Natalia Lafourcade,Hasta la Raíz
82,2015,Luke Combs,Hurricane
108,2015,Travis Scott,Antidote
178,2015,Halsey,Colors
191,2015,Fetty Wap,My Way


In [49]:
#make sure we didn't match sings with the same title by diff artists 
duplicates = df[df["grammy_winner"] == 1][
    ["artist", "title", "year"]
    ]
duplicates.sample(20)

,artist,title,year
4055,Dan + Shay,"10,000 Hours",2019
1833,Auli'i Cravalho,How Far I'll Go,2016
7029,Future,WAIT FOR U,2022
3188,Silk City,Electricity,2018
8477,Sleepy Hallow,ANXIETY,2023
5068,Chris Stapleton,You Should Probably Leave,2020
3066,21 Savage,a lot,2018
1717,Hearts & Colors,Lighthouse,2016
10420,Leon Thomas,MUTT,2025
1111,Alessia Cara,How Far I'll Go,2016


In [50]:
final_df = df[["year", "artist", "title", "streams", "grammy_winner"]]

In [51]:
final_df.head(20)

,year,artist,title,streams,grammy_winner
0,2015,Lord Huron,The Night We Met,3858784170,0
1,2015,Justin Bieber,Love Yourself,3312873799,0
2,2015,Twenty One Pilots,Stressed Out,3118770904,1
3,2015,The Weeknd,The Hills,3112731294,0
4,2015,Justin Bieber,Sorry,3023709953,0
5,2015,Major Lazer,Lean On,2760278431,0
6,2015,Charlie Puth,We Don't Talk Anymore,2689454270,0
7,2015,Lukas Graham,7 Years,2578396178,0
8,2015,Shawn Mendes,Stitches,2573625181,0
9,2015,Tame Impala,The Less I Know The Better,2546245598,0


In [52]:
final_df.to_csv("spotify_songs_with_grammy.csv", index = False)

In [53]:
normalized_grammy_pages = {
    year: normalize_text(page_text)
    for year, page_text in grammy_page_text.items()
}

print("Normalized pages:", len(normalized_grammy_pages))

Normalized pages: 12


In [58]:
def check_grammy_nominee(row):
    title = row["title_clean"]
    artist = row["artist_clean"]
    song_year = int(row["year"])

    possible_ceremony_years = [
        song_year,
        song_year + 1,
        song_year + 2
    ]

    for ceremony_year in possible_ceremony_years:
        page_text = normalized_grammy_pages.get(ceremony_year,"")

        if(
            title != ""
            and artist != ""
            and title in page_text
            and artist in page_text
        ):
            return 1
    return 0
    

In [59]:
df["grammy_nominee"] = df.apply(
    check_grammy_nominee, 
    axis=1
)

In [60]:
#make sure the winners are also nominees
df.loc[
    df["grammy_winner"] == 1,
    "grammy_nominee"
    ] = 1

In [61]:
print(df["grammy_nominee"].value_counts())
print(df["grammy_winner"].value_counts())

grammy_nominee
0    10227
1      552
Name: count, dtype: int64
grammy_winner
0    10551
1      228
Name: count, dtype: int64


In [62]:
df[
    (df["grammy_nominee"] == 1) &
    (df["grammy_winner"] == 0) 
][
    ["year", "artist", "title", "grammy_nominee", "grammy_winner"]
    ].head(30)
    

,year,artist,title,grammy_nominee,grammy_winner
1,2015,Justin Bieber,Love Yourself,1,0
7,2015,Lukas Graham,7 Years,1,0
11,2015,Mike Posner,I Took A Pill In Ibiza,1,0
12,2015,Wiz Khalifa,See You Again,1,0
16,2015,The Weeknd,Can't Feel My Face,1,0
23,2015,Ellie Goulding,Love Me Like You Do,1,0
43,2015,Rihanna,FourFiveSeconds,1,0
85,2015,Justin Bieber,Company,1,0
96,2015,Adele,All I Ask,1,0
121,2015,Ellie Goulding,Love Me Like You Do,1,0


In [63]:
print("Nominee counts:")
print(df["grammy_nominee"].value_counts())

print("\nWinner counts:")
print(df["grammy_winner"].value_counts())

Nominee counts:
grammy_nominee
0    10227
1      552
Name: count, dtype: int64

Winner counts:
grammy_winner
0    10551
1      228
Name: count, dtype: int64


In [64]:
df[
    (df["grammy_winner"] == 1) &
    (df["grammy_nominee"] == 0)
]

,year,artist,title,streams,title_clean,artist_clean,grammy_winner,grammy_nominee


In [65]:
final_df = df[
    [
        "year",
        "artist",
        "title",
        "streams",
        "grammy_nominee",
        "grammy_winner"
    ]
].copy()

final_df.head(20)

,year,artist,title,streams,grammy_nominee,grammy_winner
0,2015,Lord Huron,The Night We Met,3858784170,0,0
1,2015,Justin Bieber,Love Yourself,3312873799,1,0
2,2015,Twenty One Pilots,Stressed Out,3118770904,1,1
3,2015,The Weeknd,The Hills,3112731294,0,0
4,2015,Justin Bieber,Sorry,3023709953,0,0
5,2015,Major Lazer,Lean On,2760278431,0,0
6,2015,Charlie Puth,We Don't Talk Anymore,2689454270,0,0
7,2015,Lukas Graham,7 Years,2578396178,1,0
8,2015,Shawn Mendes,Stitches,2573625181,0,0
9,2015,Tame Impala,The Less I Know The Better,2546245598,0,0


In [66]:
final_df.to_csv(
    "spotify_songs_with_grammy.csv",
    index=False,
    encoding="utf-8-sig"
)

print(final_df.shape)

(10779, 6)
